In [1]:
import os
import glob
import numpy as np
import pandas as pd

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import lightgbm as lgb

print("Environment successfully initialized!")

Environment successfully initialized!


In [2]:
# Search for dataset file
possible_paths = (
    glob.glob('/kaggle/input/**/column*.csv', recursive=True) +
    glob.glob('/kaggle/input/**/vertebral*.csv', recursive=True) +
    glob.glob('/kaggle/input/**/*.csv', recursive=True) +
    ['column_2C_weka.csv', 'column_3C_weka.csv']
)

dataset_path = None
for path in possible_paths:
    if not any(x in os.path.basename(path).lower() for x in ['metadata', 'dict', 'schema']):
        dataset_path = path
        break

if dataset_path and os.path.exists(dataset_path):
    print(f"Loading dataset from: {dataset_path}")
    df = pd.read_csv(dataset_path)
else:
    print("Dataset path not found locally. Creating synthetic benchmark matching Vertebral Column structure...")
    np.random.seed(42)
    n = 310
    df = pd.DataFrame({
        'pelvic_incidence': np.random.uniform(30, 130, size=n),
        'pelvic_tilt': np.random.uniform(-10, 50, size=n),
        'lumbar_lordosis_angle': np.random.uniform(15, 125, size=n),
        'sacral_slope': np.random.uniform(10, 90, size=n),
        'pelvic_radius': np.random.uniform(70, 160, size=n),
        'degree_spondylolisthesis': np.random.uniform(-10, 400, size=n),
        'class': np.random.choice(['Abnormal', 'Normal'], size=n, p=[0.68, 0.32])
    })

# Standardize column headers
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(f"Dataset Shape: {df.shape}")
df.head(3)

Loading dataset from: /kaggle/input/datasets/jessanrod3/vertebralcolumndataset/Dataset_spine.csv
Dataset Shape: (310, 7)


,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,grade_of_spondylolisthesis,class_att
0,63.027817,22.552586,39.609117,40.475232,98.672917,-0.254400,Abnormal
1,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259,Abnormal
2,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317,Abnormal


In [3]:
target_col = 'class' if 'class' in df.columns else df.columns[-1]

X = df.drop(columns=[target_col]).copy()
y_raw = df[target_col].copy()

# Label Encode Target
le = LabelEncoder()
y = le.fit_transform(y_raw.astype(str))

# Scale Continuous Physical Features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train Set: {X_train.shape[0]} samples | Test Set: {X_test.shape[0]} samples")
print(f"Target Classes ({len(le.classes_)}): {list(le.classes_)}")

Train Set: 248 samples | Test Set: 62 samples
Target Classes (2): ['Abnormal', 'Normal']


In [4]:
# Configure fast CPU LightGBM model
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=15,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
)

# Predictions
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"--- MODEL ACCURACY SCORE: {acc * 100:.2f}% ---\n")
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))

--- MODEL ACCURACY SCORE: 87.10% ---

Classification Report:
               precision    recall  f1-score   support

    Abnormal       0.90      0.90      0.90        42
      Normal       0.80      0.80      0.80        20

    accuracy                           0.87        62
   macro avg       0.85      0.85      0.85        62
weighted avg       0.87      0.87      0.87        62



In [5]:
# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig_eval = go.Figure(data=go.Heatmap(
    z=cm,
    x=[f"Pred: {c}" for c in le.classes_],
    y=[f"Actual: {c}" for c in le.classes_],
    colorscale='Viridis',
    showscale=False,
    text=cm,
    texttemplate="%{text}",
    textfont={"size": 16}
))

fig_eval.update_layout(
    title="<b>Vertebral Column Diagnostics: Confusion Matrix</b>",
    template="plotly_dark",
    height=400,
    width=600
)

# 2. Feature Importance
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=True)

fig_imp = px.bar(
    importance_df,
    x='Importance',
    y='Feature',
    orientation='h',
    color='Importance',
    color_continuous_scale=['#8B5CF6', '#EC4899'],
    title="<b>Top Spine Feature Importances</b>",
    template="plotly_dark"
)
fig_imp.update_layout(height=400, showlegend=False)

# Display both figures using raw HTML injection for safe Kaggle output
display(HTML(fig_eval.to_html(include_plotlyjs='cdn', full_html=False)))
display(HTML(fig_imp.to_html(include_plotlyjs='cdn', full_html=False)))